# Tune & Ensemble a Classifier
Baseline **Decision Tree** → 5-fold CV → tuned with **GridSearchCV** and **RandomizedSearchCV** → **Random Forest** ensemble → single comparison table.

Dataset: Breast Cancer Wisconsin (`data.csv`), target = `diagnosis` (M = 1, B = 0).

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                     GridSearchCV, RandomizedSearchCV)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

## 2. Load & prepare data

In [2]:
df = pd.read_csv('data.csv')
print(df.shape)
df.head()

(569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
print(df.isnull().sum())

id                           0
diagnosis                    0
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
Unnamed:

`Unnamed: 32` is an empty trailing column and `id` is an identifier — both dropped. No other nulls.

In [4]:
df = df.drop(columns=['id', 'Unnamed: 32'])
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

X = df.drop(columns='diagnosis')
y = df['diagnosis']
print(X.shape, y.value_counts().to_dict())

(569, 30) {0: 357, 1: 212}


## 3. Train/test split (same split used for every model)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

(455, 30) (114, 30)


## 4. Baseline Decision Tree — 5-fold CV

In [6]:
baseline = DecisionTreeClassifier(random_state=RANDOM_STATE)
base_cv_scores = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='accuracy')
print('Fold scores:', np.round(base_cv_scores, 4))
print(f'Baseline CV accuracy: {base_cv_scores.mean():.4f} ± {base_cv_scores.std():.4f}')

baseline.fit(X_train, y_train)
base_test_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f'Baseline test accuracy: {base_test_acc:.4f}')

Fold scores: [0.8901 0.967  0.8901 0.9121 0.956 ]
Baseline CV accuracy: 0.9231 ± 0.0326
Baseline test accuracy: 0.9298


## 5. GridSearchCV

In [7]:
param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [3, 4, 5, 6, 8, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                    param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

grid_test_acc = accuracy_score(y_test, grid.predict(X_test))
print('Best params:', grid.best_params_)
print(f'Best CV accuracy: {grid.best_score_:.4f}')
print(f'Test accuracy:    {grid_test_acc:.4f}')

Best params: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2}
Best CV accuracy: 0.9407
Test accuracy:    0.9298


## 6. RandomizedSearchCV

In [8]:
param_dist = {
    'criterion': ['gini', 'entropy'],
    'max_depth': randint(2, 15),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10)
}

rand = RandomizedSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE),
                          param_dist, n_iter=50, cv=cv, scoring='accuracy',
                          random_state=RANDOM_STATE, n_jobs=-1)
rand.fit(X_train, y_train)

rand_test_acc = accuracy_score(y_test, rand.predict(X_test))
print('Best params:', rand.best_params_)
print(f'Best CV accuracy: {rand.best_score_:.4f}')
print(f'Test accuracy:    {rand_test_acc:.4f}')

Best params: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}
Best CV accuracy: 0.9363
Test accuracy:    0.9298


## 7. Ensemble — Random Forest (same split, same CV folds)

In [9]:
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='accuracy')
print('Fold scores:', np.round(rf_cv_scores, 4))
print(f'Random Forest CV accuracy: {rf_cv_scores.mean():.4f} ± {rf_cv_scores.std():.4f}')

rf.fit(X_train, y_train)
rf_test_acc = accuracy_score(y_test, rf.predict(X_test))
print(f'Random Forest test accuracy: {rf_test_acc:.4f}')

Fold scores: [0.956  0.989  0.9451 0.956  0.967 ]
Random Forest CV accuracy: 0.9626 ± 0.0149


Random Forest test accuracy: 0.9649


## 8. Results table

In [10]:
results = pd.DataFrame([
    {'Model': 'Decision Tree (Baseline)', 'CV Accuracy': base_cv_scores.mean(),
     'Test Accuracy': base_test_acc, 'Best Params': 'default'},
    {'Model': 'Decision Tree + GridSearchCV', 'CV Accuracy': grid.best_score_,
     'Test Accuracy': grid_test_acc, 'Best Params': str(grid.best_params_)},
    {'Model': 'Decision Tree + RandomizedSearchCV', 'CV Accuracy': rand.best_score_,
     'Test Accuracy': rand_test_acc, 'Best Params': str(rand.best_params_)},
    {'Model': 'Random Forest (Ensemble)', 'CV Accuracy': rf_cv_scores.mean(),
     'Test Accuracy': rf_test_acc, 'Best Params': 'n_estimators=200 (default otherwise)'},
])
results[['CV Accuracy', 'Test Accuracy']] = results[['CV Accuracy', 'Test Accuracy']].round(4)

pd.set_option('display.max_colwidth', None)
results

,Model,CV Accuracy,Test Accuracy,Best Params
0,Decision Tree (Baseline),0.9231,0.9298,default
1,Decision Tree + GridSearchCV,0.9407,0.9298,"{'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 2, 'min_samples_split': 2}"
2,Decision Tree + RandomizedSearchCV,0.9363,0.9298,"{'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 3, 'min_samples_split': 2}"
3,Random Forest (Ensemble),0.9626,0.9649,n_estimators=200 (default otherwise)


## 9. Conclusion
- **Baseline Decision Tree**: 0.9231 CV / 0.9298 test — highest fold-to-fold variance (±0.033), typical of an unconstrained tree.
- **GridSearchCV** (108 combos) lifted CV accuracy to 0.9407 with `max_depth=4`, entropy criterion; **RandomizedSearchCV** (50 draws) found a near-identical tree at 0.9363. Test accuracy didn't move (0.9298) — on 114 test rows, one sample = 0.88 pp, so the CV gain is the more reliable signal.
- **Random Forest** is the clear winner: 0.9626 CV (±0.015, half the baseline variance) and 0.9649 test. Averaging many decorrelated trees reduces variance in a way tuning a single tree can't.